# Import thư viện cần thiết

In [3]:
import os
import json
import random
import pandas as pd
from datasets import load_dataset, Dataset
from functools import reduce

/Users/macbook/Desktop/DS304_perfume_investigation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Set biến global

In [4]:
# Biến Global
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")  # set token qua biến môi trường
HF_REPO_TEXT_ID = "UngLong/openm3chest-labels-v2"
HF_REPO_OUTPUT  = "rimine/radiology_ready_finetune"


# Hàm load các dataset

In [5]:
def build_samples(subset_list, repo_input, task_type):
    """
    Merge cac task (inner join tren pids+keys) -> DataFrame:
    keys, pids, prompt, response (JSON), task_type.
    - Detail tasks tu loc Pool A vi inner join (chi scan co nodule moi co label detail).
    - allowed values lay dong tu answer_dict cua tung task.
    """
    dfs = []
    for i, subset in enumerate(subset_list):
        print(f"Loading {subset}...")
        df = pd.DataFrame(load_dataset(repo_input, split="train", name=subset))
        df = df.drop_duplicates(subset=["pids", "keys"], keep="first")
        df = df.rename(columns={"labels": f"label_{subset}",
                                "questions": f"ques_{subset}",
                                "answer_dict": f"ans_{subset}"})
        cols = ["keys", "pids", f"label_{subset}", f"ques_{subset}", f"ans_{subset}"]
        if i == 0:
            cols.insert(2, "clinical_data")
        dfs.append(df[cols])

    merged = reduce(lambda l, r: pd.merge(l, r, on=["pids", "keys"], how="inner"), dfs)
    print(f"[{task_type}] {len(subset_list)} tasks -> {len(merged)} scans sau inner join")

    prompts, responses = [], []
    for _, row in merged.iterrows():
        clinical = clinical_to_text(json.loads(row["clinical_data"]))

        q_lines, ans_obj = [], {}
        for subset in subset_list:
            q = random.choice(row[f"ques_{subset}"])
            ans_dict = json.loads(row[f"ans_{subset}"])
            allowed = " | ".join(ans_dict.values())
            q_lines.append(f'- "{subset}": {q} (allowed: {allowed})')
            ans_obj[subset] = ans_dict.get(str(row[f"label_{subset}"]), "Unknown")

        keys_str = ", ".join(f'"{s}"' for s in subset_list)
        prompt = f"""You are a helpful medical assistant. Analyze the provided chest CT scan slices together with the patient's clinical record.

[PATIENT CLINICAL RECORD]
{clinical}

[QUESTIONS]
For each item below, choose exactly one of its allowed values:
{chr(10).join(q_lines)}

[OUTPUT FORMAT]
Respond ONLY with a single JSON object (no extra text) using exactly these keys: {{{keys_str}}}."""

        prompts.append(prompt)
        responses.append(json.dumps(ans_obj, ensure_ascii=False))

    merged["prompt"] = prompts
    merged["response"] = responses
    merged["task_type"] = task_type
    return merged[["keys", "pids", "prompt", "response", "task_type"]]


# Luồng để xử lý và upload lên HF

## Hàm xử lý clinical text thành đoạn văn

In [6]:
import json

def _valid(v):
    """False neu la missing sentinel: None, "", "None", hoac so <= 0."""
    if v is None:
        return False
    if isinstance(v, str):
        return v not in ("", "None")
    if isinstance(v, (int, float)):
        return v > 0
    return True


def clinical_to_text(clinical_data) -> str:
    """
    Convert structured clinical JSON -> plain English paragraph.
    Bo qua missing sentinel: -1.0 (so), "None"/"" (chuoi).
    """
    if isinstance(clinical_data, str):
        clinical_data = json.loads(clinical_data)

    demo    = clinical_data.get("demo", {}) or {}
    smoking = clinical_data.get("smoking", {}) or {}
    disease = clinical_data.get("disease_his", {}) or {}
    cancer  = clinical_data.get("cancer_his", {}) or {}
    fam     = clinical_data.get("fam_lc", {}) or {}

    parts = []

    # Demographics
    if demo:
        tokens = []
        age    = demo.get("age")
        gender = demo.get("gender", "") if _valid(demo.get("gender")) else ""
        race   = demo.get("race")
        ethnic = demo.get("ethnic")
        educat = demo.get("educat")
        height = demo.get("height")
        weight = demo.get("weight")

        if _valid(age):
            tokens.append(f"{int(age)}-year-old {gender}".strip())
        if _valid(race):
            tokens.append(race)
        if _valid(ethnic) and ethnic != "Neither Hispanic nor Latino":
            tokens.append(ethnic)
        if _valid(height) and _valid(weight):
            bmi = round(weight * 0.453592 / ((height * 0.0254) ** 2), 1)
            tokens.append(f"height {int(height)}in, weight {int(weight)}lbs (BMI {bmi})")
        if _valid(educat):
            tokens.append(f"education: {educat}")
        if tokens:
            parts.append("Patient: " + ", ".join(tokens) + ".")

    # Smoking
    status = smoking.get("cigsmok", "")
    if _valid(status) and status != "Never":
        smk      = f"Smoking: {status}"
        pkyr     = smoking.get("pkyr")
        start    = smoking.get("smokeage")
        quit_age = smoking.get("age_quit")
        smokeday = smoking.get("smokeday")
        smokeyr  = smoking.get("smokeyr")
        live     = smoking.get("smokelive", "")
        work     = smoking.get("smokework", "")
        cigar    = smoking.get("cigar", "")
        pipe     = smoking.get("pipe", "")

        if _valid(pkyr):
            smk += f", {int(pkyr)} pack-years"
        if _valid(start):
            smk += f", started age {int(start)}"
        if _valid(smokeday):
            smk += f", {int(smokeday)} cigarettes/day"
        if _valid(smokeyr):
            smk += f", {int(smokeyr)} years"
        if status == "Former" and _valid(quit_age):
            smk += f", quit age {int(quit_age)}"
        if live == "Yes":
            smk += ", lives with smoker"
        if work == "Yes":
            smk += ", works with smoker"
        if cigar == "Yes":
            smk += ", cigar smoker"
        if pipe == "Yes":
            smk += ", pipe smoker"
        parts.append(smk + ".")

    # Disease history
    if disease:
        items = []
        for name, age in disease.items():
            if _valid(age):
                items.append(f"{name} (onset age {int(age)})")
            else:
                items.append(f"{name} (onset age unknown)")
        parts.append(f"Medical history: {', '.join(items)}.")

    # Cancer history
    if cancer:
        items = []
        for name, age in cancer.items():
            if _valid(age):
                items.append(f"{name} (age {int(age)})")
            else:
                items.append(name)
        parts.append(f"Cancer history: {', '.join(items)}.")

    # Family lung cancer history
    if fam:
        relatives = ", ".join(str(k) for k in fam.keys())
        parts.append(f"Family history of lung cancer: {relatives}.")

    return " ".join(parts) if parts else "No clinical data available."


## Tiến hành xử lý — Radiology 2 bước (screening + detail)

Radiology phức tạp hơn các agent khác: nó có **2 nhóm task** và **2 pool bệnh nhân**.

- **Screening (8 task)**: chest_abn_54–61 + nodule_presence → áp dụng cho **mọi scan** (Pool A + Pool B).
- **Detail (4 task)**: nodule_location/attenuation/margin/size → **chỉ Pool A** (scan có nodule).

`inner join` **tự động tách pool**: file detail chỉ chứa scan có nodule, nên merge 4 task detail → ra đúng Pool A. Không cần lọc `nodule_presence` thủ công.

Kết quả: 1 dataset gộp, **Pool A có 2 dòng** (screening + detail), **Pool B có 1 dòng** (chỉ screening). Train 1 adapter cho cả 2, inference 2 bước (screening → nếu có nodule thì detail).


In [7]:
SCREENING_TASKS = ["chest_abn_54", "chest_abn_55", "chest_abn_56", "chest_abn_57",
                   "chest_abn_58", "chest_abn_59", "chest_abn_61", "nodule_presence"]
DETAIL_TASKS    = ["nodule_location", "nodule_attenuation", "nodule_margin", "nodule_size"]

df_screen = build_samples(SCREENING_TASKS, HF_REPO_TEXT_ID, "screening")
df_detail = build_samples(DETAIL_TASKS,    HF_REPO_TEXT_ID, "detail")

combined = pd.concat([df_screen, df_detail], ignore_index=True)
combined = combined.sample(frac=1, random_state=3407).reset_index(drop=True)  # shuffle
print(f"\nScreening: {len(df_screen)} | Detail: {len(df_detail)} | Tong: {len(combined)}")

# Dataset.from_pandas(combined).push_to_hub(repo_id=HF_REPO_OUTPUT, private=False)
# print(f"Da push len: {HF_REPO_OUTPUT}")

# Xem thu
print("\n--- SCREENING SAMPLE ---")
print(df_screen.iloc[0]["prompt"])
print("RESPONSE:", df_screen.iloc[0]["response"])
print("\n--- DETAIL SAMPLE ---")
print(df_detail.iloc[0]["prompt"])
print("RESPONSE:", df_detail.iloc[0]["response"])


Loading chest_abn_54...


Generating test split: 100%|██████████| 13346/13346 [00:00<00:00, 66267.14 examples/s]


Loading chest_abn_55...


Generating test split: 100%|██████████| 12722/12722 [00:00<00:00, 165851.61 examples/s]


Loading chest_abn_56...


...

Loading nodule_margin...


Generating test split: 100%|██████████| 11620/11620 [00:00<00:00, 165075.27 examples/s]


Loading nodule_size...


Generating test split: 100%|██████████| 11552/11552 [00:00<00:00, 90314.92 examples/s]


[detail] 4 tasks -> 198 scans sau inner join

Screening: 348 | Detail: 198 | Tong: 546

--- SCREENING SAMPLE ---
You are a helpful medical assistant. Analyze the provided chest CT scan slices together with the patient's clinical record.

[PATIENT CLINICAL RECORD]
Patient: 66-year-old Male, White, height 68in, weight 175lbs (BMI 26.6), education: High school graduate/GED. Smoking: Current, 52 pack-years, started age 14, 20 cigarettes/day, 52 years, lives with smoker, works with smoker.

[QUESTIONS]
For each item below, choose exactly one of its allowed values:
- "chest_abn_54": Are there any segmental atelectasis present? (allowed: No | Yes)
- "chest_abn_55": Are there any evident changes suggesting pleural thickening or the presence of effusion? (allowed: No | Yes)
- "chest_abn_56": Does the scan display non-calcified masses or adenopathies greater than or equal to 10 mm within the hilar/mediastinal regions? (allowed: No | Yes)
- "chest_abn_57": Are there manifestations of any abnormal

In [11]:
a = len(df_screen)
b = len(df_detail)

print(a, '+', b, '=', a + b)

348 + 198 = 546


In [10]:
from datasets import load_dataset
import json
from collections import Counter
CVD = ["CVD_diagnosis", "CVD_mortality"]
ONCOL = ["lung_cancer_risk"]
RADIO = SCREENING_TASKS + DETAIL_TASKS + CVD + ONCOL  
for task in RADIO:
    ds = load_dataset(HF_REPO_TEXT_ID, split="train", name=task)
    ad = json.loads(ds[0]["answer_dict"])
    cnt = Counter(str(x) for x in ds["labels"])
    pretty = {ad.get(k, k): v for k, v in sorted(cnt.items())}
    print(f"{task:18s} (n={sum(cnt.values())}): {pretty}")


chest_abn_54       (n=348): {'No': 345, 'Yes': 3}
chest_abn_55       (n=348): {'No': 322, 'Yes': 26}
chest_abn_56       (n=348): {'No': 340, 'Yes': 8}
chest_abn_57       (n=348): {'No': 347, 'Yes': 1}
chest_abn_58       (n=348): {'No': 347, 'Yes': 1}
chest_abn_59       (n=348): {'No': 219, 'Yes': 129}
chest_abn_61       (n=348): {'No': 262, 'Yes': 86}
nodule_presence    (n=486): {'No': 150, 'Yes': 336}
nodule_location    (n=336): {'Right Upper Lobe': 90, 'Right Middle Lobe': 42, 'Right Lower Lobe': 67, 'Left Upper Lobe': 73, 'Left Lower Lobe': 64}
nodule_attenuation (n=336): {'Solid': 225, 'Ground Glass': 64, 'Others': 47}
nodule_margin      (n=336): {'Spiculated (Stellate)': 47, 'Smooth': 189, 'Poorly defined': 79, 'Unable to determine': 21}
nodule_size        (n=336): {'<=4mm': 78, '4~6mm': 122, '6~8mm': 41, '8~15mm': 54, '15~30mm': 26, '>30mm': 15}
CVD_diagnosis      (n=1500): {'No': 750, 'Yes': 750}
CVD_mortality      (n=1500): {'Low risk': 1383, 'High risk': 117}


Generating test split: 100%|██████████| 10308/10308 [00:00<00:00, 30132.24 examples/s]

lung_cancer_risk   (n=2000): {'{"y": false, "y2": 0.0, "y_seq": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "y_mask": [1, 0, 0, 0, 0, 0], "max_followup": 6, "time_at_event": 0, "candx_days": -1.0, "fup_days": 1057, "cancyr": -1.0, "can_scr": 0, "days_since_rand": 737.0, "loc_dict": {"loccar": -1.0, "loclhil": -1.0, "loclin": -1.0, "locllow": -1.0, "loclmsb": -1.0, "loclup": -1.0, "locmed": -1.0, "locoth": -1.0, "locunk": -1.0, "locrhil": -1.0, "locrlow": -1.0, "locrmid": -1.0, "locrmsb": -1.0, "locrup": -1.0}, "pid": "125567", "year": 2, "loc_idx": -1, "nodules": [], "cancer_anno": [], "cancer_slice_idx": {}, "slices": null, "loc_lr": []}': 1, '{"y": false, "y2": 0.0, "y_seq": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "y_mask": [1, 0, 0, 0, 0, 0], "max_followup": 6, "time_at_event": 0, "candx_days": -1.0, "fup_days": 1067, "cancyr": -1.0, "can_scr": 0, "days_since_rand": 707.0, "loc_dict": {"loccar": -1.0, "loclhil": -1.0, "loclin": -1.0, "locllow": -1.0, "loclmsb": -1.0, "loclup": -1.0, "locmed": -1.0, "lo